# Natural language to SQL

**Run in [Google Colab](https://colab.research.google.com/) For GPU.**

This model have  Mistral as a base and it has been fine-tuned to excel in SQL code generation.

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [2]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/accelerate.git to /tmp/pip-req-build-380sts2h
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-380sts2h
  Resolved https://github.com/huggingface/accelerate.git to commit 665444ceb62211f2b410d0d0fdb4bc013c5effdf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for accelerate: filename=accelerate-1.15.0.dev0-py3-none-any.whl size=390066 sha256=79df986ab02aef0fa7b6b359a16229f69ada30ab2938d780bd2a7b55a56f50d1
  Stored in directory: /tmp/pip-ephem-wheel-cache-3l3wly88/wheels/5a/20/fb/1221fe933b56fe7ac69fd00159d9a1950bc8ced38198abc18f
Successfully built accelerate
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [4]:
model_name = "defog/sqlcoder-7b"

We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [5]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [6]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

This function wraps the call to *model.generate*

In [8]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [9]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE 3+ TABLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """

In [10]:
sp_nl2sql = sp_nl2sql.format(question="YOUR QUERY HERE")
print(sp_nl2sql)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE 3+ TABLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `YOUR QUERY HERE`:
    ```sql3
    


In [11]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [12]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [13]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT COUNT(*) AS total_students FROM students WHERE gender = 'female' AND age >= 18 AND age <= 24;


The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [14]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

   YOUR TABLES HERE

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `{question}`:
    ```sql3
    """


In [15]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

   YOUR TABLES HERE

    ### Response
    YOUR QERIES AND SAMPLE RESPONSES HERE

    `Return The name of the best paid employee`:
    ```sql3
    


In [16]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [17]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT employees.first_name, employees.last_name, MAX(employees.salary) AS max_salary FROM employees GROUP BY employees.first_name, employees.last_name ORDER BY max_salary DESC NULLS LAST LIMIT 1;


The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [19]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE

    ### Samples

    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
    ```sql3
    """


In [20]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE
    
    ### Samples
    
    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `Return The name of the best paid employee`:
    ```sql3
    


In [21]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [22]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT employees.first_name, employees.last_name, MAX(employees.salary) AS max_salary FROM employees GROUP BY employees.first_name, employees.last_name ORDER BY max_salary DESC NULLS LAST LIMIT 1;


#Now the question in spanish.


In [23]:
sp_nl2sql3 = sp_nl2sql3b.format(question="YOUR QUERY HERE")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    YOUR TABLES HERE
    
    ### Samples
    
    YOUR SAMPLES HERE

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `YOUR QUERY HERE`:
    ```sql3
    


In [24]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [25]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

### Instructions:
Your task is function a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and;


The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Prompt Variations for SQL Generation

Here are three different prompt variations you can use to guide the model in generating SQL queries for the exercise. Each variation explores a slightly different approach to structuring the prompt, which can influence the model's output.

### Variation 1: Emphasizing Schema and Question Directly

This prompt focuses on clearly presenting the database schema and the question, with instructions to directly convert the question into SQL based on the provided schema. It's a straightforward approach that prioritizes explicit input.

In [29]:
prompt_variation_1 = """
    ### Instructions:
    Your goal is to write a SQL query to answer a given question, using the provided database schema.
    Focus on accurately translating the question into a SQL query based solely on the schema.

    ### Database Schema:
    {schema}

    ### Question:
    {question}

    ### SQL Query:
    ```sql
    """

# Example usage for Variation 1 (replace with your actual schema and question)
# schema_v1 = "CREATE TABLE employees (id INT, name VARCHAR(255), salary DECIMAL);"
# question_v1 = "What is the average salary of employees?"
# print(prompt_variation_1.format(schema=schema_v1, question=question_v1))

### Variation 2: Including an Example for Few-Shot Learning

This variation introduces a single example (a 'shot') to demonstrate the expected input-output format. This can help the model understand the desired query style or complexity, especially for specific database structures or query patterns.

In [30]:
prompt_variation_2 = """
    ### Instructions:
    Convert the natural language question into a SQL query. Pay close attention to the provided database schema and the example to understand the expected output format.

    ### Database Schema:
    {schema}

    ### Example:
    Question: What are the names of all users?
    SQL Query: SELECT name FROM users;

    ### Question:
    {question}

    ### SQL Query:
    ```sql
    """

# Example usage for Variation 2 (replace with your actual schema and question)
# schema_v2 = "CREATE TABLE users (id INT, name VARCHAR(255), email VARCHAR(255));"
# question_v2 = "How many users are there?"
# print(prompt_variation_2.format(schema=schema_v2, question=question_v2))

### Variation 3: Role-Playing and Step-by-Step Thinking

This advanced prompt instructs the model to act as a 'SQL Expert' and think step-by-step. It encourages a more deliberate processing of the request, potentially leading to more complex or accurate queries by simulating a reasoning process.

In [31]:
prompt_variation_3 = """
    ### Role:
    You are a highly skilled SQL expert. Your task is to generate precise SQL queries from natural language questions, given a database schema.

    ### Process:
    1. Carefully analyze the provided database schema.
    2. Understand the user's question completely.
    3. Formulate a step-by-step plan to construct the SQL query.
    4. Generate the final SQL query.

    ### Database Schema:
    {schema}

    ### Question:
    {question}

    ### SQL Query:
    ```sql
    """

# Example usage for Variation 3 (replace with your actual schema and question)
# schema_v3 = "CREATE TABLE orders (order_id INT, customer_id INT, order_date DATE, total_amount DECIMAL); CREATE TABLE customers (customer_id INT, customer_name VARCHAR(255));"
# question_v3 = "Show the total amount of all orders placed by customer 'John Doe' in the last month."
# print(prompt_variation_3.format(schema=schema_v3, question=question_v3))

In [32]:
def compare_prompt_outputs(schema, question):
    """
    Compares the SQL outputs from the three defined prompt variations.

    Args:
        schema (str): The database schema string.
        question (str): The natural language question to convert to SQL.

    Returns:
        dict: A dictionary containing the generated SQL queries for each prompt variation.
    """

    outputs = {}

    # Generate SQL for Variation 1
    formatted_prompt_1 = prompt_variation_1.format(schema=schema, question=question)
    inputs_1 = tokenizer(formatted_prompt_1, return_tensors="pt").to('cuda')
    response_1 = get_outputs(foundation_model, inputs_1)
    sql_1 = tokenizer.batch_decode(response_1, skip_special_tokens=True)[0].split("```sql")[-1].split("```")[0].strip()
    outputs['Variation 1'] = sql_1

    # Generate SQL for Variation 2
    formatted_prompt_2 = prompt_variation_2.format(schema=schema, question=question)
    inputs_2 = tokenizer(formatted_prompt_2, return_tensors="pt").to('cuda')
    response_2 = get_outputs(foundation_model, inputs_2)
    sql_2 = tokenizer.batch_decode(response_2, skip_special_tokens=True)[0].split("```sql")[-1].split("```")[0].strip()
    outputs['Variation 2'] = sql_2

    # Generate SQL for Variation 3
    formatted_prompt_3 = prompt_variation_3.format(schema=schema, question=question)
    inputs_3 = tokenizer(formatted_prompt_3, return_tensors="pt").to('cuda')
    response_3 = get_outputs(foundation_model, inputs_3)
    sql_3 = tokenizer.batch_decode(response_3, skip_special_tokens=True)[0].split("```sql")[-1].split("```")[0].strip()
    outputs['Variation 3'] = sql_3

    # Clear CUDA cache after each comparison
    torch.cuda.empty_cache()

    return outputs

### How to use the `compare_prompt_outputs` function

To use the function, define your database `schema` and the `question` you want to convert to SQL. Then, call the function and it will return a dictionary containing the SQL queries generated by each prompt variation.

In [33]:
# Example usage:
example_schema = """
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    name VARCHAR(255),
    department VARCHAR(255),
    salary DECIMAL(10, 2)
);

CREATE TABLE departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(255),
    location VARCHAR(255)
);
"""

example_question = "What are the names of employees who earn more than 50000 and work in the 'Sales' department?"

comparison_results = compare_prompt_outputs(example_schema, example_question)

for variation, sql_query in comparison_results.items():
    print(f"--- {variation} ---")
    print(sql_query)
    print("\n")

--- Variation 1 ---
SELECT employees.name FROM employees JOIN departments ON employees.department = departments.department_name WHERE employees.salary > 50000 AND departments.department_name ilike '%Sales%' ORDER BY employees.name NULLS LAST;


--- Variation 2 ---
SELECT employees.name FROM employees JOIN departments ON employees.department = departments.department_id WHERE employees.salary > 50000 AND departments.department_name ilike '%Sales%' ORDER BY employees.name NULLS LAST;


--- Variation 3 ---
SELECT employees.name FROM employees JOIN departments ON employees.department = departments.department_name WHERE employees.salary > 50000 AND departments.department_name ilike '%Sales%' ORDER BY employees.name NULLS LAST;


